## Main processing of the data for exploring word embeddings in the context of Ukrainian-English translation

### Installation

<i>Libraries to install with functions imports</i>

In [1]:
import fasttext
import numpy as np
import pandas as pd

from alignment import (
    learn_least_squares,
    build_candidate_matrices,
    translate_word,
    evaluate
)

from other_approaches import learn_orthogonal
from similarity_metrics import candidates_neighbours_evaluation_csls

<i>Embedding models</i>

[FastText](https://fasttext.cc/docs/en/crawl-vectors.html) embedding models for Ukrainian and English

In [8]:
model_uk = fasttext.load_model('../model/cc.uk.300.bin')
model_en = fasttext.load_model('../model/cc.en.300.bin')

### Data preparation

Creating pairs of words - Ukrainian and English

In [9]:
def data_to_dictionary(filename: str)-> dict:
    """
    Function creates from the data dictionary for 
    further usage

    :param filename: str, path to the data
    :return: dict, created dictionary
    """
    dict_res = {}
    with open(filename, "r", encoding="utf-8") as f_train:
        next(f_train)
        for line in f_train:
            line = line.strip("\n")
            ua_word, eng_word = line.split(" ")

            if ua_word not in dict_res:
                dict_res.setdefault(ua_word, [])

            dict_res[ua_word].append(eng_word)
    
    return dict_res

In [10]:
train_dict = data_to_dictionary("../data/usage/uk-en-train.csv")
test_dict = data_to_dictionary("../data/usage/uk-en-test.csv")
full_dictionary = data_to_dictionary("../data/usage/uk-en-full.csv")

In [11]:
pairs_train = []

for ua, eng_words in train_dict.items():
    for eng in eng_words:
        pairs_train.append((ua, eng))

### Main learning

Building aligned embeddings matrices for Ukrainian and English words

In [12]:
ua_embeddings, eng_embeddings = [], []

for ua, eng in pairs_train:
    ua_embeddings.append(model_uk.get_word_vector(ua))
    eng_embeddings.append(model_en.get_word_vector(eng))

A_ua, B_eng = np.column_stack(ua_embeddings), np.column_stack(eng_embeddings)

Learning the Least Squares Mapping

In [13]:
W_ls = learn_least_squares(A_ua, B_eng)

Preparing English Candidate words for translation

In [14]:
candidate_words = []
for _, eng_words in full_dictionary.items():
    for eng in eng_words:
        candidate_words.append(eng)

candidate_words, eng_candidate = build_candidate_matrices(candidate_words, model_en)
print("Number of candidate words:", len(candidate_words))

Number of candidate words: 28220


### Testing

Simple translation of a Ukrainian word "чай" with 2 different similarity metrics:

#### Cosine similarity

In [15]:
translate_word("чай", W_ls, model_uk, candidate_words, eng_candidate, None, 5, "cosine")

[('tea', 0.6168971657752991),
 ('drink', 0.5235310792922974),
 ('wine', 0.5210514664649963),
 ('coffee', 0.5137439966201782),
 ('beverage', 0.512194037437439)]

#### Cross-domain Similarity Local Scaling (CSLS)

In [16]:
r_y = candidates_neighbours_evaluation_csls(eng_candidate)
translate_word("чай", W_ls, model_uk, candidate_words, eng_candidate, r_y, 5, "csls")

[('tea', 0.11376911401748657),
 ('vinegar', -0.15535461902618408),
 ('milk', -0.1637352705001831),
 ('beverage', -0.19054454565048218),
 ('juice', -0.19130563735961914)]

### Compare the main realisation with other approach
In this block, we compare 2 different approaches for aligning Ukrainian 
and English word embeddings. Each method uses its own transformation matrix W, 
but all methods are evaluated on the same test pairs and the same English candidate words. 

In [17]:
W_orthogonal = learn_orthogonal(A_ua, B_eng)

<i>Example of Ukrainian-English pairs of words</i>

In [18]:
print(list(test_dict.items())[:5])

[('зникли', ['gone']), ('петров', ['petrov']), ('банки', ['banks', 'jars', 'cans']), ('олія', ['oil']), ('менші', ['smaller'])]


Evaluate and compare the methods

In [19]:
methods = {
    "Least squares": W_ls,
    "Orthogonal Procrustes": W_orthogonal
}
comparison_results = []

for method_name, W in methods.items():
    evaluation = evaluate(test_dict, W, model_uk, 
                        candidate_words, eng_candidate, None, top_k=5)

    comparison_results.append({
        "method": method_name,
        "Precision@1": evaluation["Precision@1"],
        "Precision@5": evaluation["Precision@5"],
        "MRR": evaluation["MRR"],
        "Mean Rank": evaluation["Mean Rank"],
    })

comparison_df = pd.DataFrame(comparison_results)
comparison_df

,method,Precision@1,Precision@5,MRR,Mean Rank
0,Least squares,0.432073,0.604342,0.499778,3.296919
1,Orthogonal Procrustes,0.457283,0.622549,0.522642,3.186275


**Least squares** achieves 42.93% Top-1 accuracy and 60.57% Top-5 accuracy. A greater method is **Orthogonal Procrustes**, with 46.08% Top-1 accuracy and 62.11% Top-5 accuracy. This suggests that **preserving the geometric structure** of the embedding space is more useful than learning an unrestricted linear mapping for the task of word translation.

### Compare the influence of different similarity metrics on the accuracy of evaluation

In [20]:
r_y_csls = candidates_neighbours_evaluation_csls(eng_candidate)

similarity_metrics = ["cosine", "csls"]
ls_results = []

for metric in similarity_metrics:
    evaluation = evaluate(test_dict, W_ls, model_uk, candidate_words, eng_candidate,
                          r_y=r_y_csls if metric == "csls" else None, top_k=5,
                          similarity_metric=metric,
    )

    ls_results.append({
        "method": "Least Squares",
        "similarity_metric": metric,
        "Precision@1": evaluation["Precision@1"],
        "Precision@5": evaluation["Precision@5"],
        "MRR": evaluation["MRR"],
        "Mean Rank": evaluation["Mean Rank"],
    })

ls_df = pd.DataFrame(ls_results)
ls_df

,method,similarity_metric,Precision@1,Precision@5,MRR,Mean Rank
0,Least Squares,cosine,0.432073,0.604342,0.499778,3.296919
1,Least Squares,csls,0.453081,0.609244,0.512208,3.258403


Based on the retrieved results, the CSLS metric is a better metric for the given task, as it not only considers the direction between vectors, as Cosine similarity does, but also reduces noise from the k-nearest neighbors of both candidates and possible translations, thereby reducing the number of words that appear in too many translations to create additional dataset noise.